In [2]:
import orekit
orekit.initVM()

from orekit.pyhelpers import setup_orekit_curdir
setup_orekit_curdir()

from org.orekit.frames import FramesFactory, TopocentricFrame, ITRFVersion
from org.orekit.bodies import OneAxisEllipsoid, GeodeticPoint, CelestialBodyFactory
from org.orekit.orbits import KeplerianOrbit, PositionAngleType
from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.utils import Constants, IERSConventions
from org.orekit.propagation.analytical import KeplerianPropagator
from org.orekit.attitudes import NadirPointing, TargetPointing
from org.hipparchus.geometry.euclidean.threed import Vector3D

from math import radians, degrees
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature


# ===============================
# PARAMETERS
# ===============================

ALTITUDE = 600000.0
INCLINATION = 97.4
SIM_DURATION = 86400.0
STEP = 10.0
ELEVATION_MASK = 10.0


# ===============================
# TIME & FRAMES
# ===============================

utc = TimeScalesFactory.getUTC()
start_date = AbsoluteDate(2026,1,1,12,0,0.0,utc)

inertialFrame = FramesFactory.getEME2000()
earthFrame = FramesFactory.getITRF(
    ITRFVersion.ITRF_2014,
    IERSConventions.IERS_2010,
    True
)

earth = OneAxisEllipsoid(
    Constants.WGS84_EARTH_EQUATORIAL_RADIUS,
    Constants.WGS84_EARTH_FLATTENING,
    earthFrame
)


# ===============================
# ORBIT
# ===============================

a = Constants.WGS84_EARTH_EQUATORIAL_RADIUS + ALTITUDE

orbit = KeplerianOrbit(
    a, 0.0, radians(INCLINATION),
    0.0, 0.0, 0.0,
    PositionAngleType.MEAN,
    inertialFrame,
    start_date,
    Constants.EGM96_EARTH_MU
)

propagator = KeplerianPropagator(orbit)


# ===============================
# GROUND STATION & SUN
# ===============================

stationPoint = GeodeticPoint(radians(15.0), radians(74.0), 30.0)
stationFrame = TopocentricFrame(earth, stationPoint, "GS")

sun = CelestialBodyFactory.getSun()


# ===============================
# ATTITUDE PROVIDERS
# ===============================

science_att = NadirPointing(inertialFrame, earth)
downlink_att = TargetPointing(inertialFrame, earth, stationFrame)
charging_att = TargetPointing(inertialFrame, earth, sun)


# ===============================
# LOGGING
# ===============================

currentDate = start_date

time_log = []
mode_log = []
lat_log = []
lon_log = []
slew_angles = []

previous_attitude = None
previous_mode = None

mode_durations = {"SCIENCE":0, "DOWNLINK":0, "CHARGING":0}
mode_switch_count = 0


# ===============================
# SUNLIGHT CHECK
# ===============================

def is_in_sunlight(state):
    sat_pos = state.getPVCoordinates(inertialFrame).getPosition()
    sun_pos = sun.getPVCoordinates(state.getDate(), inertialFrame).getPosition()

    sun_dir = sun_pos.subtract(sat_pos)
    earth_vec = sat_pos.negate()

    angle = Vector3D.angle(earth_vec, sun_dir)
    earth_ang_radius = Constants.WGS84_EARTH_EQUATORIAL_RADIUS / sat_pos.getNorm()

    return angle > earth_ang_radius


# ===============================
# MAIN LOOP
# ===============================

while currentDate.durationFrom(start_date) < SIM_DURATION:

    state = propagator.propagate(currentDate)

    satPosE = state.getPVCoordinates(earthFrame).getPosition()
    satGeo = earth.transform(satPosE, earthFrame, currentDate)

    elevation = degrees(
        stationFrame.getElevation(satPosE, earthFrame, currentDate)
    )

    in_sunlight = is_in_sunlight(state)

    # ---- MODE SELECTION ----
    if elevation > ELEVATION_MASK:
        mode = "DOWNLINK"
        propagator.setAttitudeProvider(downlink_att)

    elif in_sunlight:
        mode = "CHARGING"
        propagator.setAttitudeProvider(charging_att)

    else:
        mode = "SCIENCE"
        propagator.setAttitudeProvider(science_att)

    mode_durations[mode] += STEP

    # ---- SLEW CALCULATION ----
    attitude = state.getAttitude()

    if previous_mode is not None and mode != previous_mode:
        mode_switch_count += 1

        if previous_attitude is not None:
            rot1 = previous_attitude.getRotation()
            rot2 = attitude.getRotation()

            angle = degrees(Vector3D.angle(
                rot1.applyTo(Vector3D.PLUS_K),
                rot2.applyTo(Vector3D.PLUS_K)
            ))

            slew_angles.append(angle)

    previous_attitude = attitude
    previous_mode = mode

    # ---- LOGGING ----
    time_log.append(currentDate.durationFrom(start_date)/3600.0)
    mode_log.append(mode)
    lat_log.append(degrees(satGeo.getLatitude()))
    lon_log.append(degrees(satGeo.getLongitude()))

    currentDate = currentDate.shiftedBy(STEP)


# ===============================
# RESULTS
# ===============================

print("\nTotal Science Time (hrs/day):", mode_durations["SCIENCE"]/3600)

print("\nDaily Operations Summary:")
for k, v in mode_durations.items():
    print(f"{k}: {v/3600:.2f} hrs")

print("\nMode Switches:", mode_switch_count)
print("Average Slew Angle (deg):",
      np.mean(slew_angles) if slew_angles else 0)


# ===============================
# MODE TIMELINE
# ===============================

mode_numeric = [1 if m=="SCIENCE" else 2 if m=="CHARGING" else 3 for m in mode_log]

plt.figure()
plt.plot(time_log, mode_numeric)
plt.yticks([1,2,3], ["SCIENCE","CHARGING","DOWNLINK"])
plt.xlabel("Time (hours)")
plt.ylabel("Mode")
plt.title("Operational Timeline")
plt.grid()
plt.show()


# ===============================
# SCIENCE COVERAGE MAP
# ===============================

science_lat = [lat_log[i] for i,m in enumerate(mode_log) if m=="SCIENCE"]
science_lon = [lon_log[i] for i,m in enumerate(mode_log) if m=="SCIENCE"]
science_lon = [(lon + 180) % 360 - 180 for lon in science_lon]

plt.figure(figsize=(10,5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

hb = ax.hexbin(
    science_lon,
    science_lat,
    gridsize=60,
    cmap='inferno',
    transform=ccrs.PlateCarree()
)

plt.colorbar(hb, label="Science Sampling Density")
plt.title("Science Coverage Density – 1 Day")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [4]:
# in a notebook cell
add mvn org.orekit orekit 11.3

SyntaxError: invalid syntax (1245483345.py, line 2)